**MGMT298D: Science and Strategy of AI**
# Week 3: Reinforcement Learning & Dynamic Pricing

#### This notebook explores how reinforcement learning can be applied to dynamic pricing. We start with multi-armed bandit strategies (random, epsilon-greedy, UCB) to find the best fixed price, then extend to Q-learning where the agent learns state-dependent pricing policies using `numpy` throughout.

# 1 Setup & Pricing Environment

#### We define a set of candidate prices and their true (hidden) conversion rates. The agent's job is to figure out which price maximizes expected revenue through trial and error.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
PRICES = [5, 10, 15, 20, 25]
TRUE_CONVERSION_RATES = [0.50, 0.35, 0.22, 0.12, 0.05]

price_data = pd.DataFrame({
    'Price': PRICES,
    'Conversion Rate': TRUE_CONVERSION_RATES,
    'Expected Revenue': [p * cr for p, cr in zip(PRICES, TRUE_CONVERSION_RATES)]
})

print("Price Options and Expected Revenue:")
print(price_data.to_string(index=False))
print(f"\nOptimal arm (highest expected revenue): Price ${PRICES[np.argmax([p * cr for p, cr in zip(PRICES, TRUE_CONVERSION_RATES)])]}")

# 1.1 Multi-Armed Bandit Agent

#### The base bandit agent tracks how many times each arm (price) has been tried and its running average reward. Each strategy below extends this with a different arm-selection rule.

In [ ]:
class MultiArmedBandit:
    """Multi-armed bandit agent for dynamic pricing"""
    def __init__(self, num_arms):
        self.num_arms = num_arms
        self.counts = np.zeros(num_arms)
        self.values = np.zeros(num_arms)
        self.cumulative_reward = 0
        self.rewards_history = []
    
    def update(self, arm, reward):
        """Update arm estimates with new reward observation"""
        self.counts[arm] += 1
        old_value = self.values[arm]
        self.values[arm] = old_value + (reward - old_value) / self.counts[arm]
        self.cumulative_reward += reward
        self.rewards_history.append(self.cumulative_reward)

def simulate_purchase(arm):
    """Simulate a purchase at given price, return revenue if converted"""
    price = PRICES[arm]
    conversion_rate = TRUE_CONVERSION_RATES[arm]
    converted = np.random.rand() < conversion_rate
    return price if converted else 0

---
# 2 Bandit Strategies

#### We compare three strategies for selecting which price to show each customer: random selection (baseline), epsilon-greedy (mostly exploit, sometimes explore), and UCB (explore arms with high uncertainty).

#### Random selection picks a price uniformly at random every time — no learning at all. This is our baseline.

In [ ]:
agent_random = MultiArmedBandit(len(PRICES))
num_customers = 1000

for _ in range(num_customers):
    arm = np.random.randint(len(PRICES))
    reward = simulate_purchase(arm)
    agent_random.update(arm, reward)

print(f"Random Strategy (n={num_customers}):")
print(f"  Total Revenue: ${agent_random.cumulative_reward:.2f}")
print(f"  Avg per Customer: ${agent_random.cumulative_reward / num_customers:.2f}")

#### Epsilon-greedy exploits the best-known arm most of the time, but explores a random arm with probability epsilon.

In [ ]:
agent_eg = MultiArmedBandit(len(PRICES))
epsilon = 0.1
num_customers = 1000

for _ in range(num_customers):
    if np.random.rand() < epsilon:
        arm = np.random.randint(len(PRICES))
    else:
        arm = np.argmax(agent_eg.values)
    reward = simulate_purchase(arm)
    agent_eg.update(arm, reward)

print(f"Epsilon-Greedy (epsilon={epsilon}, n={num_customers}):")
print(f"  Total Revenue: ${agent_eg.cumulative_reward:.2f}")
print(f"  Avg per Customer: ${agent_eg.cumulative_reward / num_customers:.2f}")
print("\nLearned Values and Selection Counts:")
learned_values = pd.DataFrame({
    'Price': PRICES,
    'Learned Value': agent_eg.values,
    'Times Selected': agent_eg.counts.astype(int)
})
print(learned_values.to_string(index=False))

#### UCB selects the arm with the highest upper confidence bound — balancing the estimated value with an exploration bonus that shrinks as an arm is tried more.

In [ ]:
class UCBAgent(MultiArmedBandit):
    def __init__(self, num_arms, confidence=2.0):
        super().__init__(num_arms)
        self.confidence = confidence
    
    def select_arm(self):
        """Select arm with highest UCB value"""
        ucb_values = np.zeros(self.num_arms)
        for arm in range(self.num_arms):
            if self.counts[arm] == 0:
                ucb_values[arm] = float('inf')
            else:
                exploration_bonus = self.confidence * np.sqrt(np.log(sum(self.counts)) / self.counts[arm])
                ucb_values[arm] = self.values[arm] + exploration_bonus
        return np.argmax(ucb_values)

agent_ucb = UCBAgent(len(PRICES), confidence=2.0)
num_customers = 1000

for _ in range(num_customers):
    arm = agent_ucb.select_arm()
    reward = simulate_purchase(arm)
    agent_ucb.update(arm, reward)

print(f"UCB Strategy (confidence=2.0, n={num_customers}):")
print(f"  Total Revenue: ${agent_ucb.cumulative_reward:.2f}")
print(f"  Avg per Customer: ${agent_ucb.cumulative_reward / num_customers:.2f}")
print("\nLearned Values and Selection Counts:")
learned_values_ucb = pd.DataFrame({
    'Price': PRICES,
    'Learned Value': agent_ucb.values,
    'Times Selected': agent_ucb.counts.astype(int)
})
print(learned_values_ucb.to_string(index=False))

#### Side-by-side comparison of the three bandit strategies.

In [ ]:
summary_df = pd.DataFrame({
    'Strategy': ['Random', 'Epsilon-Greedy', 'UCB'],
    'Total Revenue': [agent_random.cumulative_reward, agent_eg.cumulative_reward, agent_ucb.cumulative_reward],
    'Avg per Customer': [agent_random.cumulative_reward / num_customers,
                         agent_eg.cumulative_reward / num_customers,
                         agent_ucb.cumulative_reward / num_customers]
})
print("Strategy Summary:")
print(summary_df.to_string(index=False))

In [ ]:
plt.plot(agent_random.rewards_history, label='Random')
plt.plot(agent_eg.rewards_history, label='Epsilon-Greedy')
plt.plot(agent_ucb.rewards_history, label='UCB')
plt.xlabel('Customer')
plt.ylabel('Cumulative Revenue ($)')
plt.title('Bandit Strategy Comparison')
plt.legend()
plt.show()

---
# 3 Q-Learning for State-Dependent Pricing

#### Bandits treat every customer the same. Q-learning extends this by conditioning on state — here, inventory level and time of day — so the agent can learn different optimal prices for different situations.

#### The pricing environment has 4 states (high/low inventory × early/late) and modifies conversion rates accordingly. The agent sees the state and picks a price.

In [ ]:
class PricingEnvironment:
    """State-dependent pricing environment for Q-learning"""
    def __init__(self):
        self.state = 0
        self.conversion_rate_modifiers = {
            0: 1.0,    # High inventory, early: baseline
            1: 0.8,    # High inventory, late: fewer customers
            2: 1.3,    # Low inventory, early: higher conversion due to scarcity
            3: 1.1     # Low inventory, late: moderate boost
        }
    
    def get_conversion_rate(self, price_idx):
        base_rate = TRUE_CONVERSION_RATES[price_idx]
        modifier = self.conversion_rate_modifiers[self.state]
        return min(base_rate * modifier, 1.0)
    
    def step(self, action):
        """Execute action and transition to next state"""
        price = PRICES[action]
        conv_rate = self.get_conversion_rate(action)
        reward = price if np.random.rand() < conv_rate else 0
        self.state = (self.state + 1) % 4
        return reward, self.state
    
    def reset(self):
        self.state = np.random.randint(4)
        return self.state

print("State Space: 4 states (inventory x time)")
print("  0: High Inventory, Early")
print("  1: High Inventory, Late")
print("  2: Low Inventory, Early")
print("  3: Low Inventory, Late")
print(f"Action Space: {len(PRICES)} prices {PRICES}")

#### The Q-learning agent maintains a Q-table (states × actions) and updates it using the TD learning rule after each interaction.

In [ ]:
class QLearningAgent:
    """Q-learning agent for dynamic pricing"""
    def __init__(self, num_states, num_actions, learning_rate=0.1, discount_factor=0.95):
        self.num_states = num_states
        self.num_actions = num_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.q_table = np.zeros((num_states, num_actions))
        self.epsilon = 0.1
    
    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state):
        """Q-learning TD update"""
        current_q = self.q_table[state, action]
        max_next_q = np.max(self.q_table[next_state])
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * max_next_q - current_q)
        self.q_table[state, action] = new_q
    
    def get_policy(self):
        return np.argmax(self.q_table, axis=1)

# 3.1 Train Q-Learning Agent

#### We run 1000 episodes of 50 steps each. The agent explores, collects rewards, and updates its Q-table after every step.

In [ ]:
window = 50
smoothed = [np.mean(episode_rewards[max(0,i-window):i+1]) for i in range(len(episode_rewards))]
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel('Avg Reward ($)')
plt.title('Q-Learning Training Progress')
plt.show()

In [ ]:
env = PricingEnvironment()
agent = QLearningAgent(num_states=4, num_actions=len(PRICES))

num_episodes = 1000
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    episode_reward = 0
    
    for step in range(50):
        action = agent.choose_action(state)
        reward, next_state = env.step(action)
        agent.update(state, action, reward, next_state)
        episode_reward += reward
        state = next_state
    
    episode_rewards.append(episode_reward)

print(f"Training complete: {num_episodes} episodes")
print(f"  Mean episode reward: ${np.mean(episode_rewards):.2f}")
print(f"  Final 100 episode avg: ${np.mean(episode_rewards[-100:]):.2f}")

# 3.2 Learned Q-Table and Policy

#### The Q-table shows the expected value of each price in each state. The learned policy extracts the best price per state.

In [ ]:
state_names = ['High Inv, Early', 'High Inv, Late', 'Low Inv, Early', 'Low Inv, Late']
q_table_df = pd.DataFrame(
    agent.q_table,
    index=state_names,
    columns=[f'${p}' for p in PRICES]
)

print("Q-Table (State-Action Values):")
print(q_table_df.round(3))

policy = agent.get_policy()
policy_df = pd.DataFrame({
    'State': state_names,
    'Optimal Price': [PRICES[p] for p in policy],
    'Q-Value': [agent.q_table[i, policy[i]] for i in range(len(policy))]
})

print("\nLearned Pricing Policy:")
print(policy_df.to_string(index=False))